<a href="https://colab.research.google.com/github/Amitkantikar/20_cagr_script/blob/main/crudebacktest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import yfinance as yf
import pandas as pd

# ==========================================
# SETTINGS
# ==========================================
SYMBOL = "CL=F"
STOP_LOSS = 1.0
TARGET = 1.0
DAYS = 500

# ==========================================
# DOWNLOAD DATA
# ==========================================
df = yf.download(
    SYMBOL,
    period=f"{DAYS}d",
    interval="1h",
    auto_adjust=False
)

# Fix columns
df.columns = df.columns.get_level_values(0)

# Remove NaN
df = df.dropna()

# Convert timezone to IST
df = df.tz_convert("Asia/Kolkata")

# ==========================================
# DAILY DATA
# ==========================================
daily = df.resample("1D").agg({
    "Open": "first",
    "Close": "last"
}).dropna()

daily["Prev_Close"] = daily["Close"].shift(1)

# Bullish Open Condition
daily["Bullish_Open"] = daily["Open"] > daily["Prev_Close"]

# ==========================================
# BACKTEST
# ==========================================
results = []

for day in daily.index:

    if not daily.loc[day, "Bullish_Open"]:
        continue

    # Find candle between 7 PM and 8 PM IST
    start_time = pd.Timestamp(day.date()).tz_localize("Asia/Kolkata") + pd.Timedelta(hours=19)
    end_time = start_time + pd.Timedelta(hours=1)

    entry_candle = df[
        (df.index >= start_time) &
        (df.index < end_time)
    ]

    # Skip if no candle found
    if entry_candle.empty:
        continue

    # Use first candle in that range
    entry_row = entry_candle.iloc[0]

    entry_time = entry_candle.index[0]
    entry_price = float(entry_row["Close"])

    stop_price = entry_price - STOP_LOSS
    target_price = entry_price + TARGET

    # Future candles after entry
    future_data = df[df.index > entry_time]

    trade_result = "No Result"

    for idx, row in future_data.iterrows():

        high = float(row["High"])
        low = float(row["Low"])

        # Target hit
        if high >= target_price:
            trade_result = "Profit"
            break

        # Stop hit
        if low <= stop_price:
            trade_result = "Loss"
            break

    if trade_result != "No Result":
        results.append(trade_result)

# ==========================================
# RESULTS
# ==========================================
profit_count = results.count("Profit")
loss_count = results.count("Loss")

total_trades = profit_count + loss_count

win_rate = (
    (profit_count / total_trades) * 100
    if total_trades > 0 else 0
)

# ==========================================
# OUTPUT
# ==========================================
print("=" * 50)
print("CRUDE OIL 7 PM IST LONG STRATEGY")
print("=" * 50)

print(f"Total Trades : {total_trades}")
print(f"Backtest days     : {DAYS}")
print(f"Risk to Reward      : {STOP_LOSS,':',TARGET}")
print(f"Profits       : {profit_count}")
print(f"Losses        : {loss_count}")
print(f"Win Rate      : {win_rate:.2f}%")

print("=" * 50)

[*********************100%***********************]  1 of 1 completed


CRUDE OIL 7 PM IST LONG STRATEGY
Total Trades : 141
Backtest days     : 500
Risk to Reward      : (1.0, ':', 1.0)
Profits       : 90
Losses        : 51
Win Rate      : 63.83%
